# Sequence Classification using RNN/LSTM
This notebook demonstrates a complete end-to-end pipeline for sequence classification (e.g., classifying whether a mathematical curve/signal is a sine wave or a cosine wave, or predicting sentiment/categories from sequences) using PyTorch.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Generate Synthetic Sequence Data
We will generate synthetic sequences consisting of sine waves (Class 0) and cosine waves (Class 1) with added Gaussian noise.

In [2]:
def generate_data(num_samples=1000, seq_len=50):
    X = []
    y = []
    t = np.linspace(0, 2 * np.pi, seq_len)

    for _ in range(num_samples):
        # Class 0: Sine wave + noise
        if np.random.rand() > 0.5:
            noise = np.random.normal(0, 0.2, seq_len)
            wave = np.sin(t) + noise
            X.append(wave)
            y.append(0)
        # Class 1: Cosine wave + noise
        else:
            noise = np.random.normal(0, 0.2, seq_len)
            wave = np.cos(t) + noise
            X.append(wave)
            y.append(1)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)

    # Reshape X to (num_samples, seq_len, input_size) -> input_size = 1 for univariate time series
    X = np.expand_dims(X, axis=-1)
    return X, y

X, y = generate_data()
print(f"X shape: {X.shape} (samples, sequence_length, features)")
print(f"y shape: {y.shape}")

X shape: (1000, 50, 1) (samples, sequence_length, features)
y shape: (1000,)


## 2. Train-Test Split and PyTorch DataLoaders

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to PyTorch Tensors
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 3. Define the LSTM Model for Sequence Classification

In [4]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # LSTM layer
        # batch_first=True means input shapes are (batch, seq_len, feature)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connected readout layer
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Initialize hidden state and cell state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # Forward propagate LSTM
        # out shape: (batch_size, seq_len, hidden_size)
        out, _ = self.lstm(x, (h0, c0))

        # Decode the hidden state of the last time step
        out = self.fc(out[:, -1, :])
        return out

# Model Parameters
INPUT_SIZE = 1
HIDDEN_SIZE = 64
NUM_LAYERS = 2
NUM_CLASSES = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
print(model)

LSTMClassifier(
  (lstm): LSTM(1, 64, num_layers=2, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


## 4. Train the Model

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

    train_loss = epoch_loss / total
    train_acc = (correct / total) * 100
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

Epoch [1/10], Loss: 0.2143, Accuracy: 88.75%
Epoch [2/10], Loss: 0.0025, Accuracy: 100.00%
Epoch [3/10], Loss: 0.0005, Accuracy: 100.00%
Epoch [4/10], Loss: 0.0003, Accuracy: 100.00%
Epoch [5/10], Loss: 0.0003, Accuracy: 100.00%
Epoch [6/10], Loss: 0.0002, Accuracy: 100.00%
Epoch [7/10], Loss: 0.0002, Accuracy: 100.00%
Epoch [8/10], Loss: 0.0001, Accuracy: 100.00%
Epoch [9/10], Loss: 0.0001, Accuracy: 100.00%
Epoch [10/10], Loss: 0.0001, Accuracy: 100.00%


## 5. Evaluate the Model

In [6]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(batch_y.numpy())

print("Classification Report:")
print(classification_report(all_targets, all_preds, target_names=['Sine Wave', 'Cosine Wave']))

print("Confusion Matrix:")
print(confusion_matrix(all_targets, all_preds))

Classification Report:
              precision    recall  f1-score   support

   Sine Wave       1.00      1.00      1.00       109
 Cosine Wave       1.00      1.00      1.00        91

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Confusion Matrix:
[[109   0]
 [  0  91]]
